### Sistema de Elasticidad Lineal

$$\begin{array}{rl}
-{\rm div}(\sigma({\bf u}))={\bf 0} & \text{ en }\Omega \\
{\bf u} = {\bf 0} & \text{ en }\Gamma_1 \\
\sigma({\bf u})\cdot \vec{n} = {\bf g} & \text{ en } \Gamma_2 \\
\sigma({\bf u})\cdot \vec{n} = {\bf 0} & \text{ en } \Gamma_3
\end{array}
$$
con $\sigma({\bf u}) = 2\mu \epsilon({\bf u}) + \lambda {\rm tr}(\epsilon ({\bf u}))I$, y $\epsilon({\bf u}) = \frac12\left( \frac{\partial u_i}{\partial x_j} + \frac{\partial u_j}{\partial x_i} \right)$, y los parámetros de Lamé:
$$\lambda = \frac{\nu E}{(1+\nu)(1-2\nu)},\quad \mu = \frac{E}{2(1+\nu)},$$
siendo $E$ el módulo de Young, y $\nu$ el coeficiente de Poisson. Ahora el material es diferente en distintas partes del dominio. 

$\Omega = \Omega_1 \cup \Omega_2$, con $\Omega_1 = (0,4)\times (0,1)\times (0,1)$, $\Omega_2=(4,8)\times(0,1)\times(0,1)$. $\lambda = \mu = 50$ en $\Omega_1$; $\lambda = \mu = 1$ en $\Omega_2$.

$ {\bf g} = (0,-10^{-2},0) \text{ en }\Gamma_2$


In [ ]:
%reset -f
import mfem.ser as mfem
from glvis import glvis

###  Malla inicial

In [ ]:
meshfile = 'mallas/beam-tet.mesh'

mesh = mfem.Mesh(meshfile)
dim = mesh.Dimension()

print(f'Número de elementos: {mesh.GetNE():4d}')
print(f'Número de vértices: {mesh.GetNV():4d}')

g = glvis(mesh)
g

#### Refinamiento Uniforme de malla

In [ ]:
ref_levels = 2
for i in range(ref_levels):
    mesh.UniformRefinement()

print(f'Número de elementos: {mesh.GetNE():4d}')
print(f'Número de vértices: {mesh.GetNV():4d}')    

g.update(mesh,keys="a")

### Espacio de elementos finitos vectorial

In [ ]:
fec = mfem.H1_FECollection(2, dim)
fespace = mfem.FiniteElementSpace(mesh, fec, dim)

print(f'Número de incógnitas del espacio de elementos finitos: {fespace.GetTrueVSize():4d}')

### Condiciones frontera

In [ ]:
print("Etiquetas de la malla: " + str(mesh.bdr_attributes.ToList()))

Condición Dirichlet ${\bf u}={\bf 0}$ en la frontera 1

Extraemos los nodos del espacio asociados a la condición frontera en `ess_tdof_list`

In [ ]:
ess_bdr = mfem.intArray([1,0,0])
ess_tdof_list = mfem.intArray()
fespace.GetEssentialTrueDofs(ess_bdr, ess_tdof_list)

### Formulación variacional 
$$\int_\Omega \left(\lambda {\rm div}({\bf u}) {\rm div}({\bf v}) + 2\mu \epsilon({\bf u}):\epsilon({\bf v})\right) =  \int_{\Gamma_2} {\bf g}\cdot {\vec n} $$

#### Fuerza aplicada
$$ {\bf g} = (0,-10^{-2},0) \text{ en }\Gamma_2$$

In [ ]:
g = mfem.VectorConstantCoefficient(mfem.Vector([0., -1.e-2, 0.]))

In [ ]:
bN = mfem.intArray([0,1,0])

b = mfem.LinearForm(fespace)
b.AddBoundaryIntegrator(mfem.VectorBoundaryLFIntegrator(g), bN)
b.Assemble()

#### Tensor elástico a trozos
La malla está dividida en dos regiones. Asignamos a una de ellas valores $\lambda$ y $\mu$ iguales a $1$, y en la otra iguales a $50$

In [ ]:
print(mesh.attributes.ToList())

In [ ]:
lambda_c = mfem.PWConstCoefficient(mfem.Vector([50.,1.]))
mu_c = mfem.PWConstCoefficient(mfem.Vector([50.,1.]))

In [ ]:
a = mfem.BilinearForm(fespace)
a.AddDomainIntegrator(mfem.ElasticityIntegrator(lambda_c, mu_c))
a.Assemble()

### Resolución
`u` es usada para pasar la condición Dirichlet, y luego para recuperar la solución como una `GridFunction`

In [ ]:
u = mfem.GridFunction(fespace)
u.Assign(0.0)

A = mfem.SparseMatrix()
B = mfem.Vector()
U = mfem.Vector()
a.FormLinearSystem(ess_tdof_list, u, b, A, U, B)
M = mfem.GSSmoother(A)

mfem.PCG(A, M, B, U, 0, 1500, 1e-8, 0.0)

a.RecoverFEMSolution(U,b,u)

### Visualización de resultados
Para visualizar el desplazamiento en la malla necesitamos mover los nodos según el deplazamiento. Para obtener los nodos vía una `GridFunction` usamos `SetNodalFESpace`, y a continuación podemos extraer los nodos con `GetNodes`

In [ ]:
deform_mesh = mfem.Mesh(mesh)
deform_mesh.SetNodalFESpace(fespace)
nodes = deform_mesh.GetNodes()
nodes += u

glvis((deform_mesh,u),keys="aamjlcc***********")


### Cálculo de la tensión de Von Misses
Definimos un `Coefficient` (clase que hereda de `PyCoefficientBase`) asociado a la función de desplazamiento que realiza el cálculo de la tensión correspondiente a través de su método `Eval`

Dicho método tiene como parámetros el elemento y los puntos de integración (`T` e `ip`) y realiza el cálculo necesario
$$\sigma_{VM} = \sqrt{ \frac{ (\sigma_{xx}-\sigma_{yy})^2 + (\sigma_{yy}-\sigma_{zz})^2 + (\sigma_{zz}-\sigma_{xx})^2 + 6( \tau_{xy}^2 + \tau_{xz}^2 + \tau_{yz}^2) } {2} }$$
donde $\sigma({\bf u}) = \begin{pmatrix} \sigma_{xx} & \tau_{xy} & \tau_{xz} \\ \tau_{xy} & \sigma_{yy} & \tau_{yz} \\ \tau_{xz} & \tau_{yz} & \sigma_{zz} \end{pmatrix}$

In [ ]:
from math import sqrt

class StressCoefficient(mfem.PyCoefficientBase):
    def __init__(self, lambda_, mu_, u):
        super(StressCoefficient, self).__init__(0)
        self.lam = lambda_   # coefficient
        self.mu = mu_       # coefficient
        self.u = u   # displacement GridFunction
        self.grad = mfem.DenseMatrix()
        self.sigma = mfem.DenseMatrix()
        
    def Eval(self, T, ip):
        L = self.lam.Eval(T, ip)
        M = self.mu.Eval(T, ip)
        self.u.GetVectorGradient(T, self.grad)
        self.grad.Symmetrize()  # grad_sym = (1/2)*(grad(u) + grad(u)^t)
        # sigma = lambda*trace(grad_sym)*I + 2*mu*grad_sym
        self.sigma.Diag(L*self.grad.Trace(), self.grad.Size()); 
        self.sigma.Add(2*M, self.grad);          
        # von misses stress
        sig = (self.sigma[0,0]-self.sigma[1,1])**2 + (self.sigma[1,1]-self.sigma[2,2])**2 \
          + (self.sigma[2,2] - self.sigma[0,0])**2 + 6* \
        (self.sigma[0,1]**2 + self.sigma[0,2]**2 + self.sigma[1,2]**2) 

        return sqrt(sig/2)

Definimos el coeficiente y proyectamos sobre él una `GridFunction`

In [ ]:
fesp = mfem.FiniteElementSpace(mesh, fec)
stress_c = StressCoefficient(lambda_c,mu_c,u)
stress = mfem.GridFunction(fesp)
stress.ProjectCoefficient(stress_c)

In [ ]:
glvis((deform_mesh,stress),keys="mjlcc***********")

### Obtención de valores de la solución en un punto

Para obtener el valor del desplazamiento en algún punto concreto usamos `FindPoints`, para encontrar la interpolación correcta de la malla, y luego el método `GetValue`

`FindPoints` permite encontrar el elemento al que pertenece un punto (dado como matriz, pues es válido para varios puntos). Lo usaremos para obtener la solución en nodos concretos de la malla

A partir del índice del elemento, podemos obtener un array con los índices de los nodos asociados al elemento

In [ ]:
count , elem, ip  = mesh.FindPoints([[2,0,0],[4,0,0],[8,0,0]])

Obtenemos el valor del vector `u` en el punto de integración asociado al elemento en el que se ha hallado el punto buscado (por interpolación)

In [ ]:
for j in range(count):    
    for i in range(dim):
        print(u.GetValue(elem[j],ip[j],i),end=' ')
    print()

In [ ]:
for j in range(count):    
    print(stress.GetValue(elem[j],ip[j]))

### Evaluación de integrales
Si queremos calcular 
$$\int_\Omega {\bf f}\cdot {\bf u}$$
teniendo en cuenta que ${\bf u} = \sum_i u_i \phi_i$, con $\phi_i$ las funciones base del espacio de elementos finitos, entonces 
$$\int_\Omega {\bf f}\cdot {\bf u} = \sum_i u_i \int_\Omega {\bf f}\cdot \phi_i$$ 
y dado que $\int_\Omega {\bf f}\cdot {\bf v}$ corresponde a la forma linea $b$, entonces $\int_\Omega {\bf f}\cdot \phi_i$ es el propio vector $B$.

In [ ]:
print(B*U)

También podemos calcular $\int_\Omega \sigma_{VM}$. Si $\sigma_{VM} =  \sum_i \alpha_i \psi_i$, pues
$$\int_\Omega \sigma_{VM} = \sum_i \alpha_i \int_\Omega \psi_i$$
Para ello creamos una `LinearForm` con coeficiente $\sigma_{VM}$

In [ ]:
s = mfem.LinearForm(fesp)
s.AddDomainIntegrator(mfem.DomainLFIntegrator(stress_c))
s.Assemble()
s.Sum()

También se puede crear la `LinearForm` a partir de una `GridFunction` con `GridFunctionCoefficient`

In [ ]:
stress_g = mfem.GridFunctionCoefficient(stress)
ss = mfem.LinearForm(fesp)
ss.AddDomainIntegrator(mfem.DomainLFIntegrator(stress_g))
ss.Assemble()
ss.Sum()                

### Otras evaluaciones
Si queremos evaluar $\int_\Omega \sigma_{VM}^2$, podemos usar algunas funciones que modifican coeficientes, en lugar de crear una nueva clase con la variante correspondiente. 

In [ ]:
sss = mfem.LinearForm(fesp)
sss.AddDomainIntegrator(mfem.DomainLFIntegrator(mfem.PowerCoefficient(stress_c,2)))
sss.Assemble()
sss.Sum()

### Cómo afectan los cambios en la función que genera el coeficiente
Si la `GridFunction` que genera el coeficiente cambia, también cambia el coeficiente, pero no la proyección de una `GridFunction` creada de antemano. Por ejemplo, si hacemos nulos los deplazamientos:

In [ ]:
u *= 0
glvis((deform_mesh,u),keys="aamjlcc***********")

Como `stress` es una proyección de un coeficiente, fue calculado con los desplazmientos originales, luego no ha cambiado.

In [ ]:
glvis((deform_mesh,stress),keys="aamjlcc***********")

Pero si ahora generamos un nueva proyección a partir del mismo coeficiente (`stress_c`) que fue creado a partir de los desplazamientos `u`, entonces

In [ ]:
stress2 = mfem.GridFunction(fesp)
stress2.ProjectCoefficient(stress_c)

glvis((deform_mesh,stress2),keys="aamjlcc***********")

Igualmente ocurre con el valor de la integral, que dependía de dicho coeficiente. Es necesario volver a ensamblar

In [ ]:
s.Assemble()
s.Sum()